Data Preparation

## 1. Load and Inspect

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

pd.set_option("display.max_columns", 50)

RAW_PATH = Path("data/raw/historical_data.csv")
OUT_DIR  = Path("data/processed")
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR  = Path("figures")
FIG_DIR.mkdir(exist_ok=True)

TEST_SIZE      = 0.20   # latest 20% of orders = test set
DELAY_PCTL     = 0.75   # delayed = duration above the 75th percentile (Proposal 6.1)
MIN_DURATION   = 5      # minutes – anything faster is treated as a data error
MAX_DURATION   = 180    # minutes – anything slower is treated as a data error (review in EDA)

In [9]:
df_raw = pd.read_csv(RAW_PATH)
print("Shape:", df_raw.shape)
df_raw.head()

Shape: (197428, 16)


,market_id,created_at,actual_delivery_time,store_id,store_primary_category,order_protocol,total_items,subtotal,num_distinct_items,min_item_price,max_item_price,total_onshift_dashers,total_busy_dashers,total_outstanding_orders,estimated_order_place_duration,estimated_store_to_consumer_driving_duration
0,1.0,2015-02-06 22:24:17,2015-02-06 23:27:16,1845,american,1.0,4,3441,4,557,1239,33.0,14.0,21.0,446,861.0
1,2.0,2015-02-10 21:49:25,2015-02-10 22:56:29,5477,mexican,2.0,1,1900,1,1400,1400,1.0,2.0,2.0,446,690.0
2,3.0,2015-01-22 20:39:28,2015-01-22 21:09:09,5477,NaN,1.0,1,1900,1,1900,1900,1.0,0.0,0.0,446,690.0
3,3.0,2015-02-03 21:21:45,2015-02-03 22:13:00,5477,NaN,1.0,6,6900,5,600,1800,1.0,1.0,2.0,446,289.0
4,3.0,2015-02-15 02:40:36,2015-02-15 03:20:26,5477,NaN,1.0,3,3900,3,1100,1600,6.0,6.0,9.0,446,650.0


In [10]:
df_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 197428 entries, 0 to 197427
Data columns (total 16 columns):
 #   Column                                        Non-Null Count   Dtype  
---  ------                                        --------------   -----  
 0   market_id                                     196441 non-null  float64
 1   created_at                                    197428 non-null  object 
 2   actual_delivery_time                          197421 non-null  object 
 3   store_id                                      197428 non-null  int64  
 4   store_primary_category                        192668 non-null  object 
 5   order_protocol                                196433 non-null  float64
 6   total_items                                   197428 non-null  int64  
 7   subtotal                                      197428 non-null  int64  
 8   num_distinct_items                            197428 non-null  int64  
 9   min_item_price                                19

In [11]:
df_raw.describe().T

,count,mean,std,min,25%,50%,75%,max
market_id,196441.0,2.978706,1.524867,1.0,2.0,3.0,4.0,6.0
store_id,197428.0,3530.510272,2053.496711,1.0,1686.0,3592.0,5299.0,6987.0
order_protocol,196433.0,2.882352,1.503771,1.0,1.0,3.0,4.0,7.0
total_items,197428.0,3.196391,2.666546,1.0,2.0,3.0,4.0,411.0
subtotal,197428.0,2682.331402,1823.093688,0.0,1400.0,2200.0,3395.0,27100.0
num_distinct_items,197428.0,2.670791,1.630255,1.0,1.0,2.0,3.0,20.0
min_item_price,197428.0,686.218470,522.038648,-86.0,299.0,595.0,949.0,14700.0
max_item_price,197428.0,1159.588630,558.411377,0.0,800.0,1095.0,1395.0,14700.0
total_onshift_dashers,181166.0,44.808093,34.526783,-4.0,17.0,37.0,65.0,171.0
total_busy_dashers,181166.0,41.739747,32.145733,-5.0,15.0,34.0,62.0,154.0


## 2. Data Quality Report

In [ ]:
quality = pd.DataFrame({
    "dtype": df_raw.dtypes.astype(str),
    "missing": df_raw.isna().sum(),
    "missing_%": (df_raw.isna().mean() * 100).round(2),
    "n_unique": df_raw.nunique(),
})
num = df_raw.select_dtypes("number")
quality["n_negative"] = (num < 0).sum().reindex(quality.index)
quality

,dtype,missing,missing_%,n_unique,n_negative
market_id,float64,987,0.50,6,0.0
created_at,object,0,0.00,180985,NaN
actual_delivery_time,object,7,0.00,178110,NaN
store_id,int64,0,0.00,6743,0.0
store_primary_category,object,4760,2.41,74,NaN
order_protocol,float64,995,0.50,7,0.0
total_items,int64,0,0.00,57,0.0
subtotal,int64,0,0.00,8368,0.0
num_distinct_items,int64,0,0.00,20,0.0
min_item_price,int64,0,0.00,2312,13.0


In [8]:
print("Duplicate rows:", df_raw.duplicated().sum())
print("Busy riders > on-shift riders:",
      (df_raw["total_busy_dashers"] > df_raw["total_onshift_dashers"]).sum())
print("Min item price > max item price:",
      (df_raw["min_item_price"] > df_raw["max_item_price"]).sum())

Duplicate rows: 0
Busy riders > on-shift riders: 40394
Min item price > max item price: 800
